# Diş Röntgeni: Çürük ve Kanal Tedavisi Tespiti (bulutta eğitim)

Bu defter modeli **ücretsiz bulut GPU'sunda** eğitir, kendi bilgisayarını hiç kullanmaz.
Hem **Kaggle** hem **Google Colab** üzerinde çalışır.

**Kaggle (önerilen — bilgisayarını kapatsan bile devam eder)**
1. Sağ panel: **Accelerator → GPU T4 x2**, **Internet → On**.
2. **Save Version → Save & Run All (Commit)** → sekmeyi kapatabilirsin.
3. Bitince *Output* sekmesinden `dis_modeli.zip` dosyasını indir.

**Google Colab**
1. **Çalışma zamanı → Tümünü çalıştır** (*Runtime → Run all*).
2. Google Drive izni sorulursa **izin ver**: bağlantı koparsa eğitim Drive'daki son noktadan devam eder.
   İstemiyorsan aşağıdaki ayarlarda `DRIVE_KAYDET` kutusunu kaldır.
3. Sekmeyi açık tut; bitince `dis_modeli.zip` iner.
   Kopan bağlantıdan sonra tekrar *Tümünü çalıştır* dersen eğitim kaldığı epoch'tan devam eder.

Eğitim T4'te **yaklaşık 2 saat** sürer. Sonunda inen zip'i bilgisayarındaki proje klasörüne çıkar
(`weights/best.pt` oluşur), sonra `python app.py` çalıştır.

Veri seti: Mureșanu, Hedeșiu, Iacob (2025), *Dataset for Automating Dental Condition Detection on Panoramic
Radiographs*, Zenodo, [doi:10.5281/zenodo.15487430](https://doi.org/10.5281/zenodo.15487430), CC-BY-4.0.
1628 panoramik röntgen, 3 uzman tarafından etiketlenmiş; ayrıca başka kliniklerden 180 röntgenlik dış doğrulama seti.

In [ ]:
# @title 1) Ayarlar
MODEL = "yolo26s.pt"  # @param ["yolo26s.pt", "yolo11s.pt", "yolov8s.pt", "yolo26m.pt"]
EPOCHS = 100  # @param {type:"integer"}
IMGSZ = 1024  # @param {type:"integer"}
DRIVE_KAYDET = True  # @param {type:"boolean"}

In [ ]:
# @title 2) GPU kontrolü ve kurulum
import os, shutil, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
if not gpu:
    raise RuntimeError("GPU yok! Menü: Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU. Sonra tekrar Tümünü çalıştır.")
print(gpu)

BASE = "/kaggle/working/dental" if os.path.exists("/kaggle/working") else "/content/dental"
os.makedirs(BASE, exist_ok=True)
os.chdir(BASE)
sys.path.insert(0, BASE)

if DRIVE_KAYDET and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DENTAL_RUNS_DIR"] = "/content/drive/MyDrive/dis_modeli/runs"
    print("Eğitim kayıtları Drive'a yazılacak:", os.environ["DENTAL_RUNS_DIR"])

!pip -q install "ultralytics>=8.4" "gradio>=6"

In [ ]:
# @title 3) Proje dosyalarını yaz (bilgisayarındaki kodun aynısı; tıklayıp içini görebilirsin)
from pathlib import Path

FILES = {
    'config.py': r'''"""Projenin ortak ayarları: yollar, sınıf isimleri, renkler."""

import os
from pathlib import Path

ROOT = Path(__file__).resolve().parent
DATA_DIR = ROOT / "data"
DATASET_DIR = DATA_DIR / "dental_xray"          # zip buraya açılır
DATA_YAML = DATA_DIR / "dental.yaml"            # train / val / test
EXTERNAL_YAML = DATA_DIR / "dental_external.yaml"  # başka kliniklerden gelen dış doğrulama seti
WEIGHTS_DIR = ROOT / "weights"
BEST_WEIGHTS = WEIGHTS_DIR / "best.pt"          # eğitim bitince en iyi model buraya kopyalanır
METRICS_FILE = WEIGHTS_DIR / "metrics.json"     # test sonuçları (uygulamada gösterilir)
EXAMPLES_DIR = ROOT / "ornekler"
# Colab'da Google Drive'a yönlendirilir; bağlantı koparsa eğitim kaldığı yerden devam eder
RUNS_DIR = Path(os.environ.get("DENTAL_RUNS_DIR", ROOT / "runs"))

# Veri seti: "Dataset for Automating Dental Condition Detection on Panoramic Radiographs"
# Mureșanu, Hedeșiu, Iacob (2025) - Zenodo, CC-BY-4.0 - https://doi.org/10.5281/zenodo.15487430
DATASET_URL = (
    "https://zenodo.org/api/records/15487430/files/"
    "panoramic_radiography_yolo_dataset_14_classes.zip/content"
)
DATASET_ZIP = DATA_DIR / "panoramic_radiography_yolo_dataset_14_classes.zip"
DATASET_MD5 = "18d5acaea021f4f3b05b76e9a27fbd42"
DATASET_SIZE = 1_397_393_531

# YOLO etiketlerindeki sınıf sırası (satırdaki ilk sayı = bu listedeki indeks)
CLASS_CODES = ["IMP", "PRR", "OBT", "END", "CAR", "BON", "IMT",
               "API", "ROT", "FUR", "APS", "ROR", "ORD", "SRD"]

TR_NAMES = {
    "IMP": "İmplant",
    "PRR": "Kron / Köprü",
    "OBT": "Dolgu",
    "END": "Kanal Tedavisi",
    "CAR": "Çürük",
    "BON": "Kemik Kaybı",
    "IMT": "Gömülü Diş",
    "API": "Periapikal Lezyon",
    "ROT": "Kök Artığı",
    "FUR": "Furkasyon Lezyonu",
    "APS": "Apikal Rezeksiyon",
    "ROR": "Kök Rezorpsiyonu",
    "ORD": "Ortodontik Aparey",
    "SRD": "Cerrahi Materyal",
}

# Varsayılan olarak ekranda gösterilen bulgular
TARGET_CODES = ["CAR", "END"]

# Çizim renkleri (RGB). Gri röntgen üzerinde kolay seçilsin diye doygun renkler.
COLORS = {
    "CAR": (255, 59, 48),     # kırmızı
    "END": (0, 190, 255),     # mavi
    "API": (255, 204, 0),     # sarı
    "OBT": (52, 199, 89),     # yeşil
    "PRR": (175, 82, 222),    # mor
    "IMP": (255, 149, 0),     # turuncu
    "IMT": (255, 45, 145),    # pembe
    "BON": (90, 200, 250),
    "ROT": (255, 214, 10),
    "FUR": (191, 90, 242),
    "APS": (100, 210, 255),
    "ROR": (255, 105, 97),
    "ORD": (48, 209, 88),
    "SRD": (152, 152, 157),
}

# Eğitim varsayılanları (6 GB ekran kartı için ayarlandı)
DEFAULT_MODEL = "yolo26s.pt"
IMGSZ = 1024

# Aynı veri setiyle yayımlanan makalenin (YOLOv8, Diagnostics 2024, doi:10.3390/diagnostics14202336)
# iç test setindeki sonuçları - kendi modelimizi kıyaslamak için
PAPER_BASELINE = {
    "test": {"CAR": {"precision": 0.626, "recall": 0.511},
             "END": {"precision": 0.879, "recall": 0.899}},
    "external": {"CAR": {"precision": 0.466, "recall": 0.439},
                 "END": {"precision": 0.734, "recall": 0.777}},
}


def tr_name(code: str) -> str:
    return TR_NAMES.get(code, code)
''',
    'prepare_data.py': r'''"""Veri setini indirir, açar ve YOLO için data yaml dosyalarını hazırlar.

Kullanım:
    python prepare_data.py

Zenodo tek bağlantıda yavaş olduğu için dosya paralel parçalar halinde indirilir.
"""

import hashlib
import json
import random
import shutil
import sys
import threading
import time
import urllib.request
import zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import config

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

PIECE = 16 * 1024 * 1024  # her parça 16 MB
WORKERS = 16


def _fetch_range(url, dest, start, end, progress, lock):
    for attempt in range(6):
        written = 0
        try:
            req = urllib.request.Request(url, headers={"Range": f"bytes={start}-{end}"})
            with urllib.request.urlopen(req, timeout=60) as r, open(dest, "r+b") as f:
                if r.status != 206:
                    raise IOError(f"sunucu parça indirmeyi desteklemedi (HTTP {r.status})")
                f.seek(start)
                while chunk := r.read(1024 * 1024):
                    f.write(chunk)
                    written += len(chunk)
                    with lock:
                        progress[0] += len(chunk)
            if written != end - start + 1:
                raise IOError("parça eksik indi")
            return
        except Exception:  # ağ hatası -> bu parçayı baştan dene
            with lock:
                progress[0] -= written
            if attempt == 5:
                raise
            time.sleep(2 + attempt * 2)


def download(url, dest: Path, size: int):
    part = dest.with_suffix(dest.suffix + ".part")
    with open(part, "wb") as f:
        f.truncate(size)
    ranges = [(s, min(s + PIECE, size) - 1) for s in range(0, size, PIECE)]
    progress, lock = [0], threading.Lock()
    done = threading.Event()

    def report():
        t0 = time.time()
        while not done.wait(15):
            mb = progress[0] / 1e6
            print(f"  {mb:7.1f} / {size / 1e6:.1f} MB  ({mb / (time.time() - t0):.1f} MB/s)", flush=True)

    threading.Thread(target=report, daemon=True).start()
    try:
        with ThreadPoolExecutor(WORKERS) as ex:
            for fut in [ex.submit(_fetch_range, url, part, s, e, progress, lock) for s, e in ranges]:
                fut.result()
    finally:
        done.set()
    part.replace(dest)


def md5sum(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        while chunk := f.read(8 * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()


def find_split_dirs(root: Path) -> dict:
    """images/ klasörlerini bulup train/val/test/external olarak eşler."""
    splits = {}
    for img_dir in sorted(p for p in root.rglob("images") if p.is_dir()):
        key = str(img_dir.relative_to(root)).replace("\\", "/").lower()
        if "ext" in key or "alte_cabinete" in key:
            splits["external"] = img_dir
        elif key.startswith("train") or "/train" in key:
            splits["train"] = img_dir
        elif "valid" in key or key.startswith("val") or "/val" in key:
            splits["val"] = img_dir
        elif "test" in key:
            splits["test"] = img_dir
    return splits


def count_labels(img_dir: Path) -> tuple[int, Counter]:
    lbl_dir = img_dir.parent / "labels"
    counts = Counter()
    n_img = sum(1 for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
    for txt in lbl_dir.glob("*.txt"):
        for line in txt.read_text(encoding="utf-8", errors="ignore").splitlines():
            if line.strip():
                counts[int(float(line.split()[0]))] += 1
    return n_img, counts


def write_yaml(path: Path, root: Path, train: Path, val: Path, test: Path | None, names: list[str]):
    rel = lambda p: p.relative_to(root).as_posix()
    lines = [f"path: {json.dumps(root.as_posix(), ensure_ascii=False)}",
             f"train: {rel(train)}",
             f"val: {rel(val)}"]
    if test is not None:
        lines.append(f"test: {rel(test)}")
    lines.append("names:")
    lines += [f"  {i}: {n}" for i, n in enumerate(names)]
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def main():
    config.DATA_DIR.mkdir(exist_ok=True)
    zip_path = config.DATASET_ZIP

    if not (zip_path.exists() and zip_path.stat().st_size == config.DATASET_SIZE):
        print(f"İndiriliyor: {config.DATASET_URL}\n  ({config.DATASET_SIZE / 1e9:.2f} GB, Zenodo, CC-BY-4.0)")
        download(config.DATASET_URL, zip_path, config.DATASET_SIZE)

    print("MD5 kontrol ediliyor...")
    if md5sum(zip_path) != config.DATASET_MD5:
        zip_path.unlink()
        sys.exit("HATA: dosya bozuk indi (MD5 tutmadı). Scripti tekrar çalıştırın.")

    if not config.DATASET_DIR.exists():
        print(f"Açılıyor -> {config.DATASET_DIR}")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(config.DATASET_DIR)

    splits = find_split_dirs(config.DATASET_DIR)
    missing = {"train", "val", "test"} - splits.keys()
    if missing:
        sys.exit(f"HATA: şu bölümler bulunamadı: {missing}. Bulunanlar: {splits}")

    root = config.DATASET_DIR
    names = config.CLASS_CODES
    write_yaml(config.DATA_YAML, root, splits["train"], splits["val"], splits["test"], names)
    if "external" in splits:
        # Ultralytics 'train' anahtarı istiyor; dış set 'val' olarak değerlendirilir.
        write_yaml(config.EXTERNAL_YAML, root, splits["train"], splits["external"], None, names)
    print(f"Yazıldı: {config.DATA_YAML.name}" + (f", {config.EXTERNAL_YAML.name}" if "external" in splits else ""))

    print(f"\n{'Bölüm':<10}{'Röntgen':>9}{'Çürük':>8}{'Kanal':>8}{'Toplam kutu':>13}")
    car, end = names.index("CAR"), names.index("END")
    for key in ("train", "val", "test", "external"):
        if key in splits:
            n_img, c = count_labels(splits[key])
            print(f"{key:<10}{n_img:>9}{c[car]:>8}{c[end]:>8}{sum(c.values()):>13}")

    # Arayüzde "örnek" olarak kullanılacak birkaç test röntgeni
    config.EXAMPLES_DIR.mkdir(exist_ok=True)
    if not any(config.EXAMPLES_DIR.iterdir()):
        test_imgs = sorted(p for p in splits["test"].iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
        for p in random.Random(0).sample(test_imgs, k=min(4, len(test_imgs))):
            shutil.copy2(p, config.EXAMPLES_DIR / p.name)
        print(f"\nÖrnek röntgenler kopyalandı -> {config.EXAMPLES_DIR.name}/")

    print("\nHazır. Sıradaki adım: python train.py")


if __name__ == "__main__":
    main()
''',
    'train.py': r'''"""YOLO modelini diş röntgeni veri setiyle eğitir, sonra test setinde değerlendirir.

Kullanım:
    python train.py                         # varsayılan model (config.DEFAULT_MODEL)
    python train.py --model yolov8s.pt      # YOLOv8 ile
    python train.py --epochs 150 --imgsz 1280
    python train.py --resume                # yarıda kalan son eğitime devam et
    python train.py --auto-resume           # yarım eğitim varsa devam et, yoksa yeni başlat (Colab)
"""

import argparse
import shutil
import sys
from pathlib import Path

import config

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")


def batch_arg(v: str):
    f = float(v)
    return int(f) if f.is_integer() else f


def unfinished_run() -> Path | None:
    """runs/ altında yarıda kalmış en yeni eğitimin last.pt dosyası (biten eğitimlerde epoch = -1 olur)."""
    from ultralytics.utils.patches import torch_load

    for last in sorted(config.RUNS_DIR.glob("*/weights/last.pt"), key=lambda p: p.stat().st_mtime, reverse=True):
        if torch_load(last, map_location="cpu").get("epoch", -1) != -1:
            return last
    return None


def main():
    ap = argparse.ArgumentParser(description="Diş röntgeni tespit modeli eğitimi")
    ap.add_argument("--model", default=config.DEFAULT_MODEL,
                    help="yolov8s.pt, yolo11s.pt, yolo26s.pt ... (n/s/m/l/x = küçükten büyüğe)")
    ap.add_argument("--epochs", type=int, default=100)
    ap.add_argument("--imgsz", type=int, default=config.IMGSZ, help="eğitim çözünürlüğü (piksel)")
    ap.add_argument("--batch", type=batch_arg, default=-1,
                    help="-1: GPU belleğine göre otomatik, 0.8: belleğin %%80'i, 8: sabit")
    ap.add_argument("--patience", type=int, default=30, help="bu kadar epoch iyileşme olmazsa dur")
    ap.add_argument("--workers", type=int, default=6)
    ap.add_argument("--cache", choices=["ram", "disk"], default=None,
                    help="ram: görüntüleri belleğe al (Colab/Linux'ta hızlandırır; Windows'ta önerilmez)")
    ap.add_argument("--fraction", type=float, default=1.0, help="verinin bir kısmıyla hızlı deneme (ör. 0.05)")
    ap.add_argument("--device", default=None, help="0 = ilk GPU, cpu = işlemci")
    ap.add_argument("--name", default=None, help="runs/ altındaki klasör adı")
    ap.add_argument("--resume", nargs="?", const="auto", default=None,
                    help="yarıda kalan eğitime devam et (isteğe bağlı: last.pt yolu)")
    ap.add_argument("--auto-resume", action="store_true",
                    help="yarım kalmış eğitim varsa ona devam et, yoksa yeni eğitim başlat")
    args = ap.parse_args()

    if not config.DATA_YAML.exists():
        sys.exit("Veri seti hazır değil. Önce: python prepare_data.py")

    from ultralytics import YOLO

    last = None
    if args.resume and args.resume != "auto":
        last = Path(args.resume)
    elif args.resume or args.auto_resume:
        last = unfinished_run()
        if last is None and args.resume:
            sys.exit(f"Devam edilecek yarım eğitim bulunamadı ({config.RUNS_DIR}).")

    if last:
        print(f"Yarıda kalan eğitime devam ediliyor: {last}")
        model = YOLO(str(last))
        model.train(resume=True)
    else:
        model = YOLO(args.model)
        model.train(
            data=str(config.DATA_YAML),
            epochs=args.epochs,
            imgsz=args.imgsz,
            batch=args.batch,
            patience=args.patience,
            workers=args.workers,
            cache=args.cache or False,
            fraction=args.fraction,
            device=args.device,
            project=str(config.RUNS_DIR),
            name=args.name or Path(args.model).stem,
            seed=0,
            plots=True,
        )

    best = Path(model.trainer.best)
    config.WEIGHTS_DIR.mkdir(exist_ok=True)
    shutil.copy2(best, config.BEST_WEIGHTS)
    print(f"\nEn iyi model kopyalandı: {best} -> {config.BEST_WEIGHTS}")

    from evaluate import evaluate
    evaluate(config.BEST_WEIGHTS, imgsz=model.trainer.args.imgsz)


if __name__ == "__main__":
    main()
''',
    'evaluate.py': r'''"""Modeli test setinde ve başka kliniklerden gelen dış doğrulama setinde ölçer.

Sonuçlar weights/metrics.json dosyasına yazılır ve arayüzde gösterilir.

Kullanım:
    python evaluate.py
    python evaluate.py --weights runs/yolo26s/weights/best.pt
"""

import argparse
import json
import sys
from pathlib import Path

import config

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")


def evaluate(weights=config.BEST_WEIGHTS, imgsz: int = config.IMGSZ, batch: int = 8) -> dict:
    from ultralytics import YOLO

    model = YOLO(str(weights))
    out = {"weights": Path(weights).name, "imgsz": imgsz, "splits": {}}
    sets = [("test", config.DATA_YAML, "test"), ("external", config.EXTERNAL_YAML, "val")]
    for key, yaml_path, split in sets:
        if not yaml_path.exists():
            continue
        m = model.val(data=str(yaml_path), split=split, imgsz=imgsz, batch=batch, plots=False,
                      verbose=False, project=str(config.RUNS_DIR), name=f"val_{key}", exist_ok=True)
        classes = {}
        for i, c in enumerate(m.box.ap_class_index):
            p, r, ap50, ap = (float(v) for v in m.box.class_result(i))
            classes[model.names[int(c)]] = {
                "precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0,
                "map50": ap50, "map50_95": ap, "instances": int(m.nt_per_class[int(c)]),
            }
        out["splits"][key] = {
            "overall": {"precision": float(m.box.mp), "recall": float(m.box.mr),
                        "map50": float(m.box.map50), "map50_95": float(m.box.map)},
            "classes": classes,
        }

    config.WEIGHTS_DIR.mkdir(exist_ok=True)
    config.METRICS_FILE.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
    print_report(out)
    return out


def print_report(metrics: dict):
    titles = {"test": "İÇ TEST SETİ (aynı klinik, eğitimde hiç görülmedi)",
              "external": "DIŞ DOĞRULAMA (başka klinikler, farklı cihazlar)"}
    for key, res in metrics["splits"].items():
        print(f"\n=== {titles.get(key, key)} ===")
        print(f"{'Bulgu':<18}{'Kesinlik':>10}{'Duyarlılık':>12}{'F1':>7}{'mAP50':>8}{'Adet':>7}   Makale (P / R)")
        for code in config.TARGET_CODES:
            c = res["classes"].get(code)
            if not c:
                continue
            base = config.PAPER_BASELINE.get(key, {}).get(code)
            ref = f"{base['precision']:.2f} / {base['recall']:.2f}" if base else "-"
            print(f"{config.tr_name(code):<18}{c['precision']:>10.3f}{c['recall']:>12.3f}{c['f1']:>7.3f}"
                  f"{c['map50']:>8.3f}{c['instances']:>7}   {ref}")
        o = res["overall"]
        print(f"{'(14 sınıf ort.)':<18}{o['precision']:>10.3f}{o['recall']:>12.3f}{'':>7}{o['map50']:>8.3f}")
    print(f"\nKaydedildi: {config.METRICS_FILE}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", default=str(config.BEST_WEIGHTS))
    ap.add_argument("--imgsz", type=int, default=config.IMGSZ)
    ap.add_argument("--batch", type=int, default=8)
    a = ap.parse_args()
    evaluate(a.weights, a.imgsz, a.batch)
''',
    'detector.py': r'''"""Model yükleme, tespit ve sonuçları röntgen üzerine çizme."""

from dataclasses import dataclass, replace
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageOps

import config


@dataclass
class Finding:
    no: int
    code: str            # veri setindeki sınıf kodu (CAR, END, ...)
    name: str            # Türkçe adı
    confidence: float    # modelin güven skoru (0-1)
    box: tuple[float, float, float, float]  # x1, y1, x2, y2 (piksel)


def load_image(source) -> Image.Image:
    """Dosya yolu, PIL görüntüsü veya numpy (RGB) dizisini RGB PIL görüntüsüne çevirir."""
    if isinstance(source, (str, Path)):
        img = Image.open(source)
    elif isinstance(source, np.ndarray):
        img = Image.fromarray(source)
    else:
        img = source
    img = ImageOps.exif_transpose(img)  # telefonla çekilen fotoğraflarda yönü düzeltir
    if img.mode in ("I;16", "I;16B", "I;16L", "I", "F"):
        # 16 bit röntgen çıktıları: 8 bite ölçekle, yoksa görüntü bembeyaz olur
        arr = np.asarray(img, dtype=np.float32)
        lo, hi = np.percentile(arr, (0.5, 99.5))
        arr = np.clip((arr - lo) / max(hi - lo, 1e-6) * 255, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)
    return img.convert("RGB")


class DentalDetector:
    def __init__(self, weights: Path | str = config.BEST_WEIGHTS, device=None):
        from ultralytics import YOLO

        weights = Path(weights)
        if not weights.exists():
            raise FileNotFoundError(
                f"Model dosyası bulunamadı: {weights}\n"
                "Önce 'python prepare_data.py' ve 'python train.py' çalıştırın."
            )
        self.model = YOLO(str(weights))
        self.device = device
        self.names = self.model.names  # {0: 'IMP', 1: 'PRR', ...}
        self.code_to_id = {code: i for i, code in self.names.items()}

    def detect(self, image, conf: float = 0.25, codes=config.TARGET_CODES,
               imgsz: int = config.IMGSZ) -> list[Finding]:
        """codes=None ise 14 sınıfın hepsi döner."""
        img = load_image(image)
        class_ids = [self.code_to_id[c] for c in codes if c in self.code_to_id] if codes else None
        result = self.model.predict(img, conf=conf, imgsz=imgsz, classes=class_ids, max_det=1000,
                                    device=self.device, verbose=False)[0]
        findings = []
        for (x1, y1, x2, y2), c, s in zip(result.boxes.xyxy.tolist(),
                                          result.boxes.cls.tolist(),
                                          result.boxes.conf.tolist()):
            code = self.names[int(c)]
            findings.append(Finding(0, code, config.tr_name(code), float(s), (x1, y1, x2, y2)))
        return select(findings)


def select(findings: list[Finding], conf: float = 0.0, codes=None) -> list[Finding]:
    """Güven eşiği ve sınıfa göre süzer; soldan sağa 1, 2, 3... diye yeniden numaralandırır.

    Arayüz modeli bir kez düşük eşikle çalıştırıp kaydırıcı değiştikçe sadece bunu çağırır.
    """
    out = [replace(f) for f in findings if f.confidence >= conf and (codes is None or f.code in codes)]
    out.sort(key=lambda f: (f.box[0] + f.box[2]) / 2)
    for i, f in enumerate(out, 1):
        f.no = i
    return out


def _font(size: int):
    for name in ("arialbd.ttf", "segoeuib.ttf", "DejaVuSans-Bold.ttf", "arial.ttf", "DejaVuSans.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            continue
    return ImageFont.load_default(size=size)


def _overlaps(a, b, pad=2):
    return not (a[2] + pad <= b[0] or b[2] + pad <= a[0] or a[3] + pad <= b[1] or b[3] + pad <= a[1])


def _shape_bounds(f: Finding, shape: str, min_r: float):
    x1, y1, x2, y2 = f.box
    if shape != "ellipse":
        return x1, y1, x2, y2
    # Kutunun köşeleri de içeride kalsın diye elipsi biraz büyüt
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    rx = max((x2 - x1) / 2 * 1.3, min_r)
    ry = max((y2 - y1) / 2 * 1.3, min_r)
    return cx - rx, cy - ry, cx + rx, cy + ry


def draw_findings(image, findings: list[Finding], shape: str = "ellipse",
                  show_labels: bool = True) -> Image.Image:
    """Bulguları elips (ya da kutu) içine alır, yanına 'No. Ad %güven' etiketi yazar."""
    img = load_image(image).copy()
    W, H = img.size
    scale = min(W, H)
    line = max(2, round(scale / 260))
    font = _font(max(12, round(scale / 45)))
    draw = ImageDraw.Draw(img)
    bounds = {id(f): _shape_bounds(f, shape, scale / 90) for f in findings}

    for f in findings:
        color = config.COLORS.get(f.code, (255, 255, 255))
        if shape == "ellipse":
            draw.ellipse(bounds[id(f)], outline=color, width=line)
        else:
            draw.rectangle(bounds[id(f)], outline=color, width=line)

    if not show_labels:
        return img

    # Etiket yeri: şeklin üstü > altı > sağı > solu; hepsi doluysa yukarı doğru kaydır
    placed = []
    pad = max(2, line)
    for f in sorted(findings, key=lambda f: -f.confidence):
        color = config.COLORS.get(f.code, (255, 255, 255))
        text = f"{f.no}. {f.name} %{f.confidence * 100:.0f}"
        tb = draw.textbbox((0, 0), text, font=font)
        tw, th = tb[2] - tb[0] + 2 * pad, tb[3] - tb[1] + 2 * pad
        sx1, sy1, sx2, sy2 = bounds[id(f)]
        cx, cy = (sx1 + sx2) / 2, (sy1 + sy2) / 2

        def clamp(lx, ly):
            lx, ly = min(max(0, lx), W - tw), min(max(0, ly), H - th)
            return lx, ly, lx + tw, ly + th

        options = [clamp(cx - tw / 2, sy1 - th - line), clamp(cx - tw / 2, sy2 + line),
                   clamp(sx2 + line, cy - th / 2), clamp(sx1 - tw - line, cy - th / 2)]
        options += [clamp(cx - tw / 2, sy1 - th - line - k * (th + line)) for k in range(1, 12)]
        idx = next((i for i, r in enumerate(options) if not any(_overlaps(r, p) for p in placed)), 0)
        rect = options[idx]
        placed.append(rect)
        if idx >= 4:  # etiket şekilden uzaklaştıysa bağlantı çizgisi çek
            draw.line(((rect[0] + rect[2]) / 2, rect[3], cx, sy1), fill=color, width=max(1, line // 2))
        draw.rectangle(rect, fill=color)
        text_color = (0, 0, 0) if sum(color) > 450 else (255, 255, 255)
        draw.text((rect[0] + pad - tb[0], rect[1] + pad - tb[1]), text, font=font, fill=text_color)
    return img


def summarize(findings: list[Finding]) -> str:
    if not findings:
        return "Seçili bulgulardan hiçbiri tespit edilmedi."
    counts = {}
    for f in findings:
        counts[f.name] = counts.get(f.name, 0) + 1
    return ", ".join(f"{n} {name}" for name, n in counts.items())
''',
    'app.py': r'''"""Web arayüzü: röntgeni yükle, çürük ve kanal tedavileri işaretlenmiş olarak gelsin.

Kullanım:
    python app.py        -> tarayıcıda http://127.0.0.1:7860 açılır
"""

import json
import os

os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")  # görüntüler/kullanım bilgisi dışarı gitmesin

import gradio as gr

import config
from detector import DentalDetector, draw_findings, load_image, select, summarize

detector = DentalDetector()

# Önce çürük ve kanal, sonra klinik olarak en ilgili olanlar
ORDER = ["CAR", "END", "API", "OBT", "PRR", "IMP", "IMT", "BON", "ROT", "FUR", "APS", "ROR", "ORD", "SRD"]
CHOICES = [(config.tr_name(c), c) for c in ORDER if c in detector.code_to_id]
SHAPES = [("Elips (yuvarlak)", "ellipse"), ("Kutu", "box")]


def analyze(image):
    """Modeli tek sefer, düşük eşikle ve tüm sınıflar için çalıştırır; süzmeyi render yapar."""
    if image is None:
        return None, []
    img = load_image(image)
    return img, detector.detect(img, conf=0.05, codes=None)


def render(img, all_findings, conf, codes, shape):
    if img is None:
        return None, "", []
    findings = select(all_findings, conf, codes or [])
    out = draw_findings(img, findings, shape=shape)
    rows = [[f.no, f.name, f"%{f.confidence * 100:.0f}"] for f in findings]
    return out, f"**Sonuç:** {summarize(findings)}", rows


def metrics_markdown() -> str:
    if not config.METRICS_FILE.exists():
        return "Henüz ölçüm yok. `python evaluate.py` çalıştırın."
    m = json.loads(config.METRICS_FILE.read_text(encoding="utf-8"))
    set_names = {"test": "İç test (aynı klinik)", "external": "Dış test (başka klinikler)"}
    lines = ["| Bulgu | Test seti | Kesinlik | Duyarlılık | mAP50 |", "|---|---|---|---|---|"]
    for code in config.TARGET_CODES:
        for key, res in m["splits"].items():
            c = res["classes"].get(code)
            if c:
                lines.append(f"| {config.tr_name(code)} | {set_names.get(key, key)} | %{c['precision'] * 100:.0f} "
                             f"| %{c['recall'] * 100:.0f} | %{c['map50'] * 100:.0f} |")
    lines += [
        "",
        "- **Kesinlik:** Model \"çürük\" dediğinde ne sıklıkla haklı çıkıyor.",
        "- **Duyarlılık:** Gerçekte var olan çürüklerin yüzde kaçını yakalıyor.",
        "- **mAP50:** Tüm eşik değerleri üzerinden genel tespit başarısı.",
        "- Görüntüdeki **%** değeri, modelin o işaretten ne kadar emin olduğudur (güven skoru); "
        "tek başına \"doğru olma ihtimali\" anlamına gelmez.",
    ]
    return "\n".join(lines)


with gr.Blocks(title="Diş Röntgeni Analizi") as demo:
    gr.Markdown(
        "# Diş Röntgeni Analizi\n"
        "Panoramik röntgeni yükleyin: model **çürükleri** ve **kanal tedavili dişleri** yuvarlak içine alır "
        "ve her birinin yanına güven skorunu yazar."
    )
    st_img = gr.State(None)
    st_findings = gr.State([])

    with gr.Row():
        with gr.Column(scale=2):
            inp = gr.Image(label="Röntgen", type="pil", image_mode=None, sources=["upload", "clipboard"])
            conf = gr.Slider(0.05, 0.95, value=0.25, step=0.05, label="Güven eşiği",
                             info="Düşürürsen daha çok (ama daha şüpheli) işaret çıkar.")
            codes = gr.CheckboxGroup(CHOICES, value=config.TARGET_CODES, label="Gösterilecek bulgular")
            shape = gr.Radio(SHAPES, value="ellipse", label="İşaret şekli")
            examples = sorted(p for p in config.EXAMPLES_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
            if examples:
                gr.Examples([[str(p)] for p in examples], inputs=inp, label="Örnek röntgenler")
        with gr.Column(scale=3):
            out = gr.Image(label="Sonuç", type="pil", format="jpeg", interactive=False)
            summary = gr.Markdown()
            table = gr.Dataframe(headers=["No", "Bulgu", "Güven"], datatype=["number", "str", "str"],
                                 interactive=False, label="Bulgular (soldan sağa)")

    with gr.Accordion("Model ne kadar doğru?", open=False):
        gr.Markdown(metrics_markdown())

    gr.Markdown(
        "> ⚠️ Bu araç tanı koymaz; yalnızca karar desteği amaçlıdır. "
        "Sonuçlar mutlaka diş hekimi tarafından değerlendirilmelidir."
    )

    view_inputs = [st_img, st_findings, conf, codes, shape]
    view_outputs = [out, summary, table]
    inp.change(analyze, inp, [st_img, st_findings]).then(render, view_inputs, view_outputs)
    for ctrl in (conf, codes, shape):
        ctrl.change(render, view_inputs, view_outputs)


if __name__ == "__main__":
    demo.launch(inbrowser=True)
''',
    'requirements.txt': r'''# Önce PyTorch'u ekran kartı (CUDA) destekli kurun:
#   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
# Sonra:
#   pip install -r requirements.txt
ultralytics>=8.4
gradio>=6
pillow
numpy
''',
}
for name, content in FILES.items():
    Path(name).write_text(content, encoding="utf-8")
print("Yazıldı:", ", ".join(FILES))

In [ ]:
# @title 4) Veri setini indir ve hazırla (~2 dk)
!python prepare_data.py

In [ ]:
# @title 4.5) Hızlı deneme (~3 dk): 1 epoch + ölçüm, her şey çalışıyor mu?
!python train.py --epochs 1 --fraction 0.05 --imgsz 640 --name deneme --cache ram

import shutil
import config
shutil.rmtree(config.RUNS_DIR / "deneme", ignore_errors=True)  # deneme çıktıları asıl eğitimi karıştırmasın
shutil.rmtree(config.WEIGHTS_DIR, ignore_errors=True)
print("\nDeneme bitti, çıktıları temizlendi. Yukarıdaki tabloda sayılar çıktıysa her şey yolunda.")

In [ ]:
# @title 5) Eğitim (~2 saat; bağlantı koparsa bu hücre kaldığı yerden devam eder)
!python train.py --auto-resume --model {MODEL} --epochs {EPOCHS} --imgsz {IMGSZ} --cache ram

In [ ]:
# @title 6) Sonuçlar: test başarısı ve örnek röntgenler
import json
from IPython.display import Image as IPImage, display

import config
from detector import DentalDetector, draw_findings, load_image, summarize
from evaluate import print_report

print_report(json.loads(config.METRICS_FILE.read_text(encoding="utf-8")))
run = max(config.RUNS_DIR.glob("*/results.png"), key=lambda p: p.stat().st_mtime).parent
display(IPImage(filename=str(run / "results.png"), width=1000))

det = DentalDetector()
for p in sorted(config.EXAMPLES_DIR.glob("*.jpg")):
    img = load_image(p)
    found = det.detect(img, conf=0.25)
    print(f"\n{p.name}: {summarize(found)}")
    out = draw_findings(img, found)
    display(out.resize((1400, round(out.height * 1400 / out.width))))

In [ ]:
# @title 7) Modeli indir (dis_modeli.zip)
import os, shutil, zipfile
from pathlib import Path

import config

if not config.BEST_WEIGHTS.exists():  # oturum koptuysa Drive'daki bitmiş eğitimden al
    from evaluate import evaluate
    best = max(config.RUNS_DIR.glob("*/weights/best.pt"), key=lambda p: p.stat().st_mtime)
    config.WEIGHTS_DIR.mkdir(exist_ok=True)
    shutil.copy2(best, config.BEST_WEIGHTS)
    evaluate(config.BEST_WEIGHTS, imgsz=IMGSZ)

run = max(config.RUNS_DIR.glob("*/results.png"), key=lambda p: p.stat().st_mtime).parent
zip_path = Path(BASE).parent / "dis_modeli.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    z.write(config.BEST_WEIGHTS, "weights/best.pt")
    z.write(config.METRICS_FILE, "weights/metrics.json")
    for f in sorted(run.iterdir()):
        if f.suffix in {".png", ".jpg", ".csv", ".yaml"}:
            z.write(f, f"egitim_grafikleri/{f.name}")
print(f"Hazır: {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)")

if IN_COLAB:
    if os.environ.get("DENTAL_RUNS_DIR"):
        shutil.copy2(zip_path, "/content/drive/MyDrive/dis_modeli/dis_modeli.zip")
        print("Drive'a da kopyalandı: MyDrive/dis_modeli/dis_modeli.zip")
    from google.colab import files
    files.download(str(zip_path))
else:  # Kaggle: veri setini sil ki kaydedilen çıktı küçük kalsın
    shutil.rmtree(config.DATASET_DIR, ignore_errors=True)
    config.DATASET_ZIP.unlink(missing_ok=True)
    print("Output sekmesinden dis_modeli.zip dosyasını indirebilirsin.")

### İsteğe bağlı: arayüzü burada dene
Aşağıdaki hücre bilgisayarında açılacak arayüzün aynısını bu sayfada açar. Link herkese açık değildir.

In [ ]:
# @title 8) Arayüzü Colab içinde aç
if IN_COLAB:
    import app
    app.demo.launch(share=False)
else:
    print("Bu hücre yalnızca Colab'da çalışır.")